# KV Cache

我们前面已经学习了Transformer的架构了，今天我们针对他的自回归部分的功能，分析其中计算的冗余性，以及如何使用Cache的思想来对自回归生成的过程进行一个加速

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/646577898)

## 理论分析

我们已经知道了，Transformer在自回归采样过程中，是把已知序列的嵌入X输入解码块，然后取出最后一个token对应的嵌入，输入一个线性头映射为下一个词为某一个词的概率，选择概率最高的token作为下一个生成词，再拼接这个词的嵌入到X，一直循环直到**生成完成(终止符)**

我们以“咕咕嘎嘎”四个字为例，看看如何进行生成的:

步骤1:

<div align="center">
    <img src="./imgs/wo_kv_1.png" alt="wo_kv_1" style="width: 450px; height: auto;">
</div>

接下来我们**省略中间步骤**，只看Q/K/V生成部分和最后拼接的结果，蓝色表示K/V**新增的部分**:

步骤2:

<div align="center">
    <img src="./imgs/wo_kv_2.png" alt="wo_kv_2" style="width: 450px; height: auto;">
</div>

步骤3:

<div align="center">
    <img src="./imgs/wo_kv_3.png" alt="wo_kv_3" style="width: 450px; height: auto;">
</div>

...

最终结果:

<div align="center">
    <img src="./imgs/wo_kv_end.png" alt="wo_kv_end" style="width: 450px; height: auto;">
</div>

我们重点关注中间的K/V新增的部分，大家发现了吗，每一次自回归生成的时候，计算需要的K/V除了最后一行的蓝色是新增的部分，它之前的行都是跟上一步计算的K/V的结果**是一样的**，也就是说，我们的自回归计算过程中获取K/V时的计算有**冗余**，如果我们可以把之前计算的K/V矩阵缓存下来，在计算新的token时，我们**只需要计算新的token产生的K/V，再与缓存的K/V矩阵进行拼接得到新的K/V矩阵，再进行后续的计算**，就能够节省很多的时间，这就是我们的K/V缓存

或许你会问: 为什么我们不对之前的Q矩阵进行缓存呢？这是因为，我们生成下一个token只需要序列中最后一个token的计算结果，不需要前面计算的Q矩阵了，它们**跟下一个生成的token无关**


那么使用KV Cache的方法，我们的计算过程就变成了如下所示:

步骤1:

<div align="center">
    <img src="./imgs/w_kv_1.png" alt="w_kv_1" style="width: 450px; height: auto;">
</div>

步骤2:

<div align="center">
    <img src="./imgs/w_kv_2.png" alt="w_kv_2" style="width: 500px; height: auto;">
</div>

步骤3:

<div align="center">
    <img src="./imgs/w_kv_3.png" alt="w_kv_3" style="width: 500px; height: auto;">
</div>

...

最终结果:

<div align="center">
    <img src="./imgs/w_kv_end.png" alt="w_kv_end" style="width: 500px; height: auto;">
</div>

上面的new QKV指的是最新的那个token计算出的新的Q/K/V。仔细观察就会发现，实际上就是多了个**缓存K/V的步骤，以及在计算时把新的K/V拼接到缓存矩阵的最后一行的操作，其他都基本不变**，实现起来很容易


## 代码实现

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, Tuple
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, dim=512, num_heads=8):
        super().__init__()

        self.dim = dim 
        self.num_heads = num_heads
        self.d_k = dim//num_heads

        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
    
    def forward(self, x:torch.Tensor, past_kv:Optional[tuple[torch.Tensor, torch.Tensor]]=None):
        bs, seq_len, _ = x.shape 

        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        q = q.view(bs, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = k.view(bs, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = v.view(bs, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # ------------------------------------------------------------------------------------------------
        # KV Cache实现部分
        # ------------------------------------------------------------------------------------------------
        if past_kv is not None:
            # past_k, past_v: [bs, seq_len-1, num_heads, d_k]
            k_cache, v_cache = past_kv
            # -> [bs, seq_len, num_heads, d_k]
            k = torch.cat([k_cache, k], dim=2)
            v = torch.cat([v_cache, v], dim=2)

        new_kv_cache = (k, v)
        

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        out = (
            out.transpose(1, 2)
            .contiguous()
            .view(bs, -1, self.dim)
        )
        out = self.wo(out)

        return out, new_kv_cache


In [ ]:
from c_kv_cache import timer


# --------------------------------------------------------------------------------------------
# 带KV Cache的自回归生成逻辑(假设只有多头自注意力，生成过程进行了简化)
# --------------------------------------------------------------------------------------------
@timer
def inference_with_kv_cache(mha:MultiHeadAttention, token_num=500, batch_size=4):
    dim = mha.dim
    tokens = torch.randn(batch_size, 1, dim)  # 初始1个token
    past_kv_cache = None 

    for i in range(token_num):
        if past_kv_cache is None:
            x = tokens 
        else:
            x = tokens[:, -1:, :]  # 只取最后一个token，下一个词只依赖于他

        out, past_kv_cache = mha(x, past_kv_cache)

        tokens = torch.cat([tokens, out[:, -1, :].unsqueeze(1)], dim=1)  # 实际还有一个线性头映射为概率再采样


In [ ]:
# --------------------------------------------------------------------------------------------
# 不带KV Cache的自回归生成逻辑(假设只有多头自注意力，生成过程进行了简化)
# --------------------------------------------------------------------------------------------
@timer
def inference_without_kv_cache(mha:MultiHeadAttention, token_num=500, batch_size=4):
    dim = mha.dim
    tokens = torch.randn(batch_size, 1, dim)

    for i in range(token_num):
        x = tokens 
        out, _ = mha(x)

        tokens = torch.cat([tokens, out[:, -1, :].unsqueeze(1)], dim=1)


In [27]:
mha = MultiHeadAttention()

print('with kv cache:', end=' ')
inference_with_kv_cache(mha)

print('without kv cache:', end=' ')
inference_without_kv_cache(mha)


with kv cache: Time consuming:  1.331551s
without kv cache: Time consuming:  10.722708s


可以看到，使用了KV Cache之后，生成的速度快了非常多